In [21]:
import requests
import pandas as pd
from datetime import timedelta

StatementMeta(, 2ff3c8f0-88a2-4ae9-a03e-05cd9fb8fb2b, 23, Finished, Available, Finished)

In [22]:
# Base URL for GitHub raw CSV files
base_url = "https://raw.githubusercontent.com/mikailaltundas/datasets-for-training/main/wind-power-dataset/"

# Path to the wind_power table in the Bronze Lakehouse
bronze_table_path = "abfss://WindPowerAnalytics@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Bronze.Lakehouse/Tables/dbo/wind_power"

StatementMeta(, 2ff3c8f0-88a2-4ae9-a03e-05cd9fb8fb2b, 24, Finished, Available, Finished)

In [23]:
# Load existing wind_power data and convert to Pandas
df_spark = spark.read.format("delta").load(bronze_table_path)
df_pandas = df_spark.toPandas()

StatementMeta(, 2ff3c8f0-88a2-4ae9-a03e-05cd9fb8fb2b, 25, Finished, Available, Finished)

In [24]:
# Find the most recent date and calculate next day's date
most_recent_date = pd.to_datetime(df_pandas['date'], format = '%Y%m%d').max()
next_date = (most_recent_date + timedelta(days = 1)).strftime('%Y%m%d')

StatementMeta(, 2ff3c8f0-88a2-4ae9-a03e-05cd9fb8fb2b, 26, Finished, Available, Finished)

In [25]:
# Download and load new data in a Pandas DataFrame
file_url = f"{base_url}{next_date}_wind_power_data.csv"
df_pandas_new = pd.read_csv(file_url)
df_pandas_new['date'] = pd.to_datetime(df_pandas_new['date'])

StatementMeta(, 2ff3c8f0-88a2-4ae9-a03e-05cd9fb8fb2b, 27, Finished, Available, Finished)

In [26]:
# Convert to Spark DataFrame and append in wind_power table
df_spark_new = spark.createDataFrame(df_pandas_new, schema = df_spark.schema)
df_spark_new.write.format("delta").mode("append").save(bronze_table_path)

StatementMeta(, 2ff3c8f0-88a2-4ae9-a03e-05cd9fb8fb2b, 28, Finished, Available, Finished)